# Clase 10 — Criterio: validez, sesgo, reproducibilidad y el revisor 2

**Deep Learning para Cientistas Sociales** · la última

> **La idea de hoy en una línea.** Hacer andar el modelo es lo fácil y lo barato.
> Demostrar **qué es lo que mide** es el trabajo profesional, y es tuyo: no hay
> una biblioteca que lo haga.

---

**Hoy no entrenamos nada nuevo. Hoy auditamos.**

Vamos a agarrar un clasificador que anda —uno que cualquiera publicaría— y le
vamos a hacer todo lo que un revisor desconfiado le haría. Vas a ver que
**el modelo no falla en ningún momento**: hace exactamente lo que le pedimos.
Lo que falla es la etiqueta sustantiva que le colgamos encima.

**Cuando termines este notebook vas a poder:**

1. Mirar qué está usando un modelo y decidir si eso es una **señal del concepto**
   o un **atajo**.
2. Correr los tests de validez que un paper serio tiene que reportar
   —ablación, control positivo, placebo, pares mínimos— y saber cuál de ellos
   sirve para qué. Y medir **el error correcto** antes de interpretar cualquier
   diferencia.
3. **Medir** el sesgo de un modelo con un procedimiento sistemático y producir
   una tabla reportable, sin exagerar lo que esa tabla prueba.
4. Generar automáticamente el bloque de reproducibilidad y la ficha del modelo
   que van al apéndice de tu paper.

**Lo que hace falta traer:** las clases 4 (partición y sobreajuste) y 9
(métricas y matriz de confusión). Nada más.

**Cómo se usa este notebook.** Corré las celdas en orden (`Shift + Enter`).
Donde diga `TODO` te toca a vos, y varios de esos TODO **no son de código**:
son de criterio, que es exactamente lo que vinimos a practicar.
Corre entero en CPU, sin GPU y sin internet, en menos de un minuto.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el repo
y deja todo a mano; en tu máquina no hace prácticamente nada. Corrémela y seguí
de largo.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso (o usa el .zip que subas) e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys
import zipfile

REPO = "https://github.com/johannperez/dlss"
RAMA = "main"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


def raiz_desde_zip():
    """Plan B: si subiste el .zip del repo (dlss-main.zip o parecido), lo abre."""
    for archivo in sorted(pathlib.Path(".").glob("dlss*.zip")):
        with zipfile.ZipFile(archivo) as z:
            z.extractall(CARPETA + "-zip")
        for ssdl in pathlib.Path(CARPETA + "-zip").glob("**/_shared/ssdl"):
            return ssdl.parent.parent.resolve()
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        clon = subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                               REPO, CARPETA],
                              env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if clon.returncode != 0:
            print("No pude clonar el repo. Pruebo con un .zip subido a mano...")
    raiz = buscar_raiz(CARPETA) if pathlib.Path(CARPETA).exists() else None
    raiz = raiz or raiz_desde_zip()

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab, si el clone falló (por ejemplo, «Repository not found» o 404):\n"
        "  el repo github.com/johannperez/dlss tiene que ser público. Si no lo es,\n"
        "  o no hay red hacia GitHub, conseguí el .zip del repo (en la página del\n"
        "  repo: Code → Download ZIP; o pedíselo a quien da el curso), subilo con\n"
        "  el panel de archivos de Colab (el nombre tiene que empezar con `dlss`,\n"
        "  como dlss-main.zip) y volvé a correr esta celda.\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`.\n"
        "Más detalle en recursos/setup.md."
    )

# Dependencias. En Colab casi todas vienen puestas: instalamos sólo lo que falte.
# En tu máquina no instalamos nada (torch es pesado y varias clases no lo usan):
# si falta algo, avisamos y lo instalás vos (ver recursos/setup.md).
faltan = []
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan and en_colab():
    print(f"Instalando {', '.join(faltan)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=True)
elif faltan:
    print(f"[!] Falta {', '.join(faltan)}. Si este notebook lo usa, instalalo con:\n"
          f"    {sys.executable} -m pip install {' '.join(faltan)}")

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports y semilla

Fijamos la semilla en 1983 **siempre**. No hace al modelo mejor: hace a tu
resultado reproducible, que es condición para poder afirmar algo. Hoy vamos a
ver, con un experimento, hasta qué punto eso importa.

In [ ]:
import hashlib
import itertools
import math
import platform
import re
import sys
from collections import Counter
from datetime import date

import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ssdl import datos, graficos as g, utils

SEMILLA = 1983
utils.fijar_semilla(SEMILLA)
utils.resumen_del_entorno()

---

## 1. La idea

Un modelo que anda te contesta **una** pregunta: *¿acierta?*. Eso se llama
**desempeño**, y se mide con las métricas de la clase 9.

Pero tu paper afirma otra cosa. Tu paper dice *"medimos la conflictividad del
discurso opositor"*, y esa afirmación necesita contestar otra pregunta:
**¿esto mide lo que yo digo que mide?** Eso se llama **validez**, y no hay
ninguna métrica que te la dé. Se demuestra con **intervenciones**: le hacés
cosas a los datos y al diseño, y mirás si el número sobrevive.

**La analogía de ciencias sociales.** Esto no es vocabulario nuevo: es
literalmente la validez de constructo de Cronbach y Meehl, de 1955, aplicada a
un instrumento nuevo. Ustedes tienen setenta años de ventaja sobre la
conversación técnica. Un clasificador es **un instrumento de medición**, como
una escala de actitudes: se reporta cómo se construyó, su confiabilidad, su
validez y sus límites. Nadie publicaría una escala sin su alfa. Nadie debería
publicar un clasificador sin su matriz de confusión sobre una muestra validada
a mano.

**Lo que NO es.** No es "el modelo se equivocó". En todo lo que sigue el modelo
va a funcionar perfectamente bien. La que se equivoca es la frase que escribimos
al lado del número.

---

## 2. Miralo: el modelo que vamos a auditar

Usamos el corpus de prensa del curso: **1471 notas reales en castellano**
(*Spanish Fake News Corpus 2.0*, CC BY 4.0), con titular, texto, medio y tema.
Son **dos recolecciones pegadas**: 971 notas juntadas de enero a julio de 2018
(año de elección presidencial en México) y 500 de noviembre de 2020 a marzo de
2021 (la pandemia). La procedencia completa está en `datos/README.md` §1.3.

La tarea es una que hacés todo el tiempo aunque no la llames así: **armar un
subcorpus temático**. En concreto, *¿esta nota es sobre política?*

Y elegimos a propósito un modelo **interpretable**: una regresión logística
sobre bolsa de palabras. No es por nostalgia. Es porque **poder mirar por dentro
es una razón legítima para elegir un modelo más simple**, y hoy vamos a
usar esa capacidad todo el tiempo. Que sea simple no lo hace válido — eso lo
vamos a ver en la sección 4 — pero lo hace **auditable**.

> **Apostá antes de correr.** ¿Qué palabras creés que va a usar un clasificador
> de "esto es sobre política" entrenado con prensa hispanoamericana de 2018 y 2020-21?
> Escribí tres en un papel. Después miramos.

In [ ]:
# --- los datos -------------------------------------------------------------
notas = datos.titulares_prensa(con_texto=True)

# Cada documento es el titular más el arranque del cuerpo. Recortamos a 600
# caracteres para que el vocabulario no explote y el notebook corra rápido.
documentos = [n["titular"] + " . " + n["texto"][:600] for n in notas]

# La etiqueta: 1 = la nota está codificada como tema "Politica" en el corpus.
# OJO CON ESTO, que vuelve en la sección 5: esa codificación NO la hicimos
# nosotros. La hizo el equipo que armó el corpus, con SU definición de política.
y = np.array([int(n["tema"] == "Politica") for n in notas])

prevalencia = float(y.mean())
piso = max(prevalencia, 1 - prevalencia)

print(f"{len(notas)} notas de prensa REALES en castellano.")
print(f"De cada 100 notas, {prevalencia * 100:.0f} están codificadas como política.")
print(f"EL PISO: un modelo que dice siempre «no es política» acierta {piso * 100:.1f} %.")
print("\nTemas del corpus:")
for tema, cuantas in Counter(n["tema"] for n in notas).most_common(6):
    print(f"   {tema:<16}{cuantas:>5}")

In [ ]:
# --- el modelo, en cinco líneas -------------------------------------------
i_train, i_test = train_test_split(
    np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEMILLA)


def entrenar(corpus, indices, etiquetas=y, min_df=2):
    """Bolsa de palabras binaria + regresión logística. Devuelve (vectorizador, modelo)."""
    vec = CountVectorizer(min_df=min_df, binary=True)
    A = vec.fit_transform([corpus[i] for i in indices])
    mod = LogisticRegression(max_iter=3000, class_weight="balanced",
                             random_state=SEMILLA)
    mod.fit(A, etiquetas[indices])
    return vec, mod


def evaluar(vec, mod, corpus, indices, etiquetas=y):
    """Las métricas de la clase 9 sobre los índices que le pases."""
    pred = mod.predict(vec.transform([corpus[i] for i in indices]))
    return g.metricas_binarias(etiquetas[indices], pred)


vec, modelo = entrenar(documentos, i_train)
metricas = evaluar(vec, modelo, documentos, i_test)

print(f"Vocabulario: {len(vec.get_feature_names_out())} palabras.")
print(f"Entrenado con {len(i_train)} notas, evaluado sobre {len(i_test)} que nunca vio.\n")
g.frase_de_metricas(y[i_test], modelo.predict(vec.transform([documentos[i] for i in i_test])),
                    fenomeno="notas sobre política")
print(f"\nGanancia real sobre el piso: {metricas['accuracy'] - piso:+.3f}")

In [ ]:
ax = g.matriz_de_confusion(
    y[i_test], modelo.predict(vec.transform([documentos[i] for i in i_test])),
    etiquetas=["no es política", "es política"],
    nombre_positivo="nota sobre política",
    titulo="El modelo que vamos a auditar")

Ahí está. Un modelo que **anda**: le gana al piso, la matriz de confusión no
tiene nada raro, y las cuatro frases en castellano se pueden decir en voz alta
sin ponerse colorado. Con este panel adelante, esto se publica.

**El resto del notebook es la parte que casi nadie hace.**

---

## 3. Hacelo vos · Test 1: ¿qué está usando el modelo?

Primera pregunta, y la más barata de todas: **mirá los términos que más pesan**.
En una regresión logística eso es literalmente leer los coeficientes.

Lo que buscás no es "términos raros". Lo que buscás es la diferencia entre:

- una **señal del concepto**: una palabra que aparece porque el texto trata del
  tema (*gobierno*, *ley*, *congreso*);
- un **atajo**: una palabra que separa las clases en *este* corpus por razones
  que no tienen nada que ver con el concepto (un nombre propio, un medio, una
  coyuntura).

Los dos suben el F1 igual. Sólo uno de los dos sobrevive fuera de tu corpus.

In [ ]:
pesos = modelo.coef_[0]
vocabulario = np.array(vec.get_feature_names_out())
orden = np.argsort(-pesos)

print("LOS 20 TÉRMINOS QUE MÁS EMPUJAN HACIA «es política»")
print("-" * 58)
for i in orden[:20]:
    print(f"   {vocabulario[i]:<18}{pesos[i]:+.2f}")

print("\nLOS 12 QUE MÁS EMPUJAN HACIA «no es política»")
print("-" * 58)
for i in orden[-12:]:
    print(f"   {vocabulario[i]:<18}{pesos[i]:+.2f}")

In [ ]:
# La misma lista, dibujada. En rojo, rayados y con ★, los que marcamos a mano
# como nombres propios: la decisión de qué es un atajo la toma una persona, no
# un algoritmo. (El rayado y la ★ son para que se lea también sin color.)
NOMBRES_PROPIOS = {"amlo", "meade", "morena", "anaya", "epn", "salinas", "pri",
                   "peña", "bronco", "sánchez", "putin", "irán", "cienfuegos",
                   "pp", "lópez", "obrador", "trump", "biden",
                   "lopezobrador_"}   # la ficha que deja la cuenta @lopezobrador_

top = [(str(vocabulario[i]), float(pesos[i])) for i in orden[:16]][::-1]
es_nombre = [t in NOMBRES_PROPIOS for t, _ in top]

fig, ax = g.figura(8.5, 5.4, titulo="Qué empuja hacia «esto es política»",
                   xlabel="peso en la regresión logística")
barras = ax.barh([("★ " if n else "") + t for (t, _), n in zip(top, es_nombre)],
                 [w for _, w in top],
                 color=[g.ERROR if n else g.MODELO for n in es_nombre])
for b, n in zip(barras, es_nombre):
    if n:
        b.set_hatch("//")
        b.set_edgecolor("white")
ax.text(0.98, 0.04, "en rojo, rayados y con ★: nombres propios de la coyuntura",
        transform=ax.transAxes, ha="right", color=g.ERROR, fontsize=10,
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"));

### `TODO 1` — leelos, uno por uno

Mirá la lista de arriba **de verdad**, no en diagonal, y contestá por escrito:

**(a)** ¿Cuántos de los 20 son **nombres propios** de personas o partidos?

**(b)** Contá (te adelanto: son **seis de veinte**, uno de cada tres: *pri*,
*lópez*, *meade*, *amlo*, *morena* y *lopezobrador_*, todos de la política
mexicana alrededor de la elección de 2018). Y contá también los otros catorce:
*presidencial*, *gobierno*, *presidente*, *electoral*, *candidato*… son
vocabulario político genérico. Con las dos cuentas a la vista, escribí **la
hipótesis** más incómoda que te parezca sostenible sobre qué está midiendo este
modelo. *Hipótesis*, no conclusión: leer coeficientes te dice dónde mirar, no
qué pasa. En la sección 7 la vamos a poner a prueba con una intervención.

**(c)** Ahora la incómoda: mirá los tres términos que anotaste en el papel antes
de correr la celda. ¿Coinciden con lo que salió? Si no coinciden, eso no es un
problema tuyo: es exactamente la razón por la que este test existe.

*Pista para (b): probá empezar por "este clasificador podría estar detectando…"
y terminar la oración sin usar la palabra «política». Después escribí qué
resultado de un experimento te haría abandonar esa hipótesis.*

**Tu respuesta:** _(escribí acá (a), (b) y (c))_

---

## 4. Antes de medir nada: **cuánto se mueve solo**

Acá viene la trampa en la que cae la mitad de los papers que usan estas
herramientas, y es un problema de estadística básica, no de deep learning.

Vamos a hacer una intervención (sacarle palabras al corpus) y comparar el F1
antes y después. Para que esa comparación signifique algo hay que saber
**cuánto se mueve el F1 sin hacer nada**, sólo por cambiar la semilla con la que
partimos los datos.

Es el mismo razonamiento de siempre: **no podés interpretar una diferencia sin
conocer su error**. Acá lo medimos por fuerza bruta, entrenando 25 veces. (Y en
la sección 5 vamos a ver que hay que medir **el error de la comparación que
hacés**, que no es el mismo que el de un F1 suelto.)

> **Apostá.** El F1 del modelo es 0,68. Si repito todo cambiando sólo la semilla
> de la partición, ¿cuánto creés que se va a mover? ¿±0,005? ¿±0,03? ¿±0,10?

In [ ]:
# 25 particiones distintas del MISMO corpus, con el MISMO modelo.
# Lo único que cambia es la semilla. Tarda unos segundos.
PARTICIONES = [train_test_split(np.arange(len(y)), test_size=0.25,
                                stratify=y, random_state=s) for s in range(25)]
efes = []
with utils.medir_tiempo("las 25 particiones"):
    for tr, te in PARTICIONES:
        v_s, m_s = entrenar(documentos, tr)
        efes.append(evaluar(v_s, m_s, documentos, te)["f1"])

efes = np.array(efes)
print(f"F1 medio  : {efes.mean():.3f}")
print(f"desvío    : {efes.std():.3f}")
print(f"del peor al mejor: {efes.min():.3f} a {efes.max():.3f}   "
      f"(un rango de {efes.max() - efes.min():.3f})")
print("\nEso es cuánto se mueve el F1 de UN modelo según cómo partiste los datos.")
print("Para reportar ese F1, se reporta con su desvío (0,68 ± 0,03), no sólo 0,68.")
print("Ojo: todavía NO es el error de una COMPARACIÓN entre dos versiones. Eso, en la 5.")

---

## 5. Test 2: la ablación

**La idea.** Si el modelo depende de un atajo, sacale el atajo del corpus,
reentrenalo y mirá cuánto cae. La caída sería la evidencia.

Es el test que recomienda toda la literatura. Lo vamos a hacer dos veces: primero
como lo hace casi todo el mundo (una sola partición) y después **bien**, midiendo
el error de la diferencia.

In [ ]:
def quitar(corpus, palabras):
    """Borra esas palabras del corpus, respetando los límites de palabra."""
    if not palabras:
        return list(corpus)
    patron = re.compile(r"\b(" + "|".join(re.escape(p) for p in palabras) + r")\b", re.I)
    return [patron.sub(" ", t) for t in corpus]


ABLACIONES = [
    ("nada (el original)", []),
    ("el término n.º 1", [str(vocabulario[orden[0]])]),
    ("los 5 primeros", [str(vocabulario[i]) for i in orden[:5]]),
    ("los 20 primeros", [str(vocabulario[i]) for i in orden[:20]]),
    ("los nombres propios", sorted(NOMBRES_PROPIOS))]
corpus_ablado = {etiqueta: quitar(documentos, quitados) for etiqueta, quitados in ABLACIONES}

# Primero, como lo hace casi todo el mundo: UNA partición (la de la sección 2).
ablaciones = []
for etiqueta, quitados in ABLACIONES:
    corpus2 = corpus_ablado[etiqueta]
    v2, m2 = entrenar(corpus2, i_train)
    f1 = evaluar(v2, m2, corpus2, i_test)["f1"]
    ablaciones.append((etiqueta, len(quitados), f1))

print("CON UNA SOLA PARTICIÓN")
print(f"{'qué le sacamos al corpus':<26}{'palabras':>9}{'F1':>8}{'contra el original':>20}")
print("-" * 64)
base_f1 = ablaciones[0][2]
for etiqueta, n_q, f1 in ablaciones:
    print(f"{etiqueta:<26}{n_q:>9}{f1:>8.3f}{f1 - base_f1:>+20.3f}")
print("\n¿Sacarle al modelo sus 5 mejores palabras lo MEJORA? Con una partición no se sabe.")

### La comparación bien hecha: la diferencia **pareada**

La tentación es comparar esas diferencias con el rango de la sección 4 (0,107)
y concluir "todo es ruido". **Es comparar mal.** El original y la versión ablada
se evalúan sobre **la misma** partición: buena parte del ruido de la sección 4
(qué notas cayeron en el test) les toca igual a los dos y se cancela en la
resta. El error que importa es el **de la diferencia**.

Así que repetimos **la misma intervención** con cada una de las 25 particiones,
y miramos la distribución de las 25 diferencias (ablado − original, cada una
sobre su propia partición). Se reportan con su media, su desvío y un intervalo,
no con el rango: el rango sólo crece si agregás semillas.

Un detalle de rigor: las 25 particiones salen del mismo corpus y comparten notas,
así que no son 25 repeticiones independientes. El error estándar ingenuo
(desvío / √25) es optimista; usamos la corrección de Nadeau y Bengio (2003),
que lo agranda (acá, unas tres veces).

In [ ]:
# La MISMA intervención, repetida sobre las 25 particiones de la sección 4.
# Tarda unos segundos (100 entrenamientos más).
n_tr, n_te = len(PARTICIONES[0][0]), len(PARTICIONES[0][1])
factor_nb = math.sqrt(1 / len(PARTICIONES) + n_te / n_tr)   # Nadeau y Bengio (2003)

pareadas = {}
with utils.medir_tiempo("las ablaciones pareadas"):
    for etiqueta, quitados in ABLACIONES[1:]:
        corpus2 = corpus_ablado[etiqueta]
        deltas = []
        for (tr, te), f0 in zip(PARTICIONES, efes):
            v2, m2 = entrenar(corpus2, tr)
            deltas.append(evaluar(v2, m2, corpus2, te)["f1"] - f0)
        pareadas[etiqueta] = np.array(deltas)

print(f"{'qué le sacamos':<22}{'Δ medio':>9}{'desvío':>8}{'IC 95 % (corregido)':>22}{'bajó en':>10}")
print("-" * 71)
for etiqueta, d in pareadas.items():
    ee = d.std(ddof=1) * factor_nb
    print(f"{etiqueta:<22}{d.mean():>+9.3f}{d.std(ddof=1):>8.3f}"
          f"{'[' + format(d.mean() - 2 * ee, '+.3f') + ', ' + format(d.mean() + 2 * ee, '+.3f') + ']':>22}"
          f"{(d < 0).sum():>6} de 25")

print(f"\nDesvío del F1 suelto (sección 4): {efes.std():.3f}.  "
      f"Desvío de la diferencia pareada: entre "
      f"{min(d.std(ddof=1) for d in pareadas.values()):.3f} y "
      f"{max(d.std(ddof=1) for d in pareadas.values()):.3f}.")

In [ ]:
# El dibujo que hay que mirar: las 25 diferencias pareadas de cada ablación,
# con su media y su intervalo. La línea vertical es "no cambió nada".
fig, ax = g.figura(9, 4.2, titulo="Cuánto cambia el F1 al sacar palabras: 25 particiones, diferencia pareada",
                   xlabel="cambio en el F1 (versión ablada − original, sobre la misma partición)")
etiquetas_abl = list(pareadas)
for k, etiqueta in enumerate(etiquetas_abl):
    d = pareadas[etiqueta]
    yk = len(etiquetas_abl) - k
    ee = d.std(ddof=1) * factor_nb
    ax.plot(d, yk + np.linspace(-0.18, 0.18, len(d)), "o", color=g.VERDAD, ms=4.5, alpha=0.8)
    ax.errorbar([d.mean()], [yk], xerr=[[2 * ee], [2 * ee]], fmt="D", color=g.PARAMETRO,
                ms=8, capsize=5, lw=2.2, zorder=3)
    una = [f1 for e, _, f1 in ablaciones if e == etiqueta][0] - base_f1
    ax.plot([una], [yk + 0.32], "v", color=g.ERROR, ms=8, zorder=4)
ax.axvline(0, color=g.DATOS, ls=":", lw=1.5)
ax.set_yticks(range(1, len(etiquetas_abl) + 1))
ax.set_yticklabels(etiquetas_abl[::-1], fontsize=10)
ax.plot([], [], "o", color=g.VERDAD, ms=5, label="cada partición (25)")
ax.plot([], [], "D", color=g.PARAMETRO, ms=7, label="media ± 2 EE (corregido)")
ax.plot([], [], "v", color=g.ERROR, ms=7, label="lo que decía UNA partición")
ax.legend(loc="upper left", fontsize=9, framealpha=0.95);

### `TODO 2` — leé las dos tablas y el gráfico

**(a)** Con **una** partición: ¿cuánto se movió el F1 al sacarle los cinco
términos más influyentes? ¿Para qué lado? Escribí la frase que habrías puesto en
el paper con sólo ese número.

**(b)** Con las **25 diferencias pareadas**: ¿qué ablaciones se distinguen de
cero (su intervalo no toca el 0) y cuáles no? ¿La lectura de una sola partición
(el triángulo rojo) se sostiene?

**(c)** Sacar los 20 términos que más pesan le cuesta al modelo unas tres
centésimas de F1. **¿Podés concluir que el modelo casi no usa atajos?**

*Pista para (c).* Cuidado con la trampa lógica. Que el F1 caiga poco puede
significar dos cosas muy distintas: (i) que no había atajo, o (ii) que hay
**muchas palabras redundantes** y al taparle unas el modelo agarra las
siguientes. En una bolsa de palabras con seis mil columnas, ¿cuál te parece más
probable? Y fijate qué **no** mide la ablación: mide cuánto pierde el modelo
**en su propio corpus**, no si lo que aprendió sirve afuera. Escribí qué test
distinguiría entre (i) y (ii). (Uno está en la sección 7.)

**Tu respuesta:** _(escribí acá)_

> **La moraleja de esta sección, que vale para tu tesis entera:** no interpretes
> una diferencia sin su error, **y medí el error de la comparación que hacés**.
> Si las dos versiones se evalúan sobre los mismos datos, repetí la comparación
> con varias particiones y reportá la diferencia pareada con su desvío o su
> intervalo, no el rango del número suelto. Con una sola partición, acá habrías
> publicado que sacarle al modelo sus mejores palabras lo mejora. Son diez
> segundos de cómputo.

---

## 6. Test 3: leé los falsos positivos

Los aciertos no te enseñan nada: el modelo hizo lo que esperabas. **El
conocimiento está en los errores**, porque cada error te muestra qué entendió
el modelo por tu categoría, y —esto es lo que sorprende— muchas veces te muestra
que **el equivocado eras vos** cuando escribiste la definición.

Esto es análisis cualitativo sobre errores cuantitativos, y es exactamente lo
que un cientista social hace mejor que nadie en un equipo de ML.

In [ ]:
proba = modelo.predict_proba(vec.transform([documentos[i] for i in i_test]))[:, 1]
pred = (proba >= 0.5).astype(int)

falsos_positivos = [(k, i) for k, i in enumerate(i_test) if pred[k] == 1 and y[i] == 0]
falsos_negativos = [(k, i) for k, i in enumerate(i_test) if pred[k] == 0 and y[i] == 1]

# Los ordenamos por confianza: los errores en los que el modelo estaba MÁS
# seguro son los más informativos.
falsos_positivos.sort(key=lambda p: -proba[p[0]])

print(f"{len(falsos_positivos)} falsos positivos y {len(falsos_negativos)} falsos negativos.\n")
print("FALSOS POSITIVOS — el modelo dijo «política» y la etiqueta dice que no")
print("=" * 96)
for k, i in falsos_positivos[:20]:
    print(f"[{proba[k]:.2f}] [{notas[i]['tema']:<12}] {notas[i]['titular'][:70]}")

### `TODO 3` — armá tu tipología de errores

Leé esos 20 errores **uno por uno** y clasificalos en categorías **tuyas**.
Las cuatro de abajo son un punto de partida; agregá o cambiá las que quieras:

- `etiqueta` — el modelo tiene razón y la etiqueta del corpus está mal
  (una nota sobre una decisión de gobierno codificada como "Sociedad").
- `frontera` — el caso es genuinamente ambiguo: es política *y* otra cosa
  (política sanitaria, política deportiva).
- `atajo` — el modelo se colgó de una palabra suelta que no venía al caso.
- `ironia` — sátira o humor que el modelo leyó literal.

Completá el diccionario de abajo y corré la celda. **Esa tabla es un resultado
publicable**: la tipología de errores de un instrumento de medición es material
de la sección de resultados, no del apéndice.

In [ ]:
# TODO 3: poné acá tu clasificación de los 20 falsos positivos.
# El índice es el número de orden en la lista de arriba (0 a 19).
MI_TIPOLOGIA = {
    # 0: "etiqueta",
    # 1: "frontera",
    # 2: "atajo",
}

if not MI_TIPOLOGIA:
    print("Todavía no clasificaste ninguno. Volvé arriba, leelos, y anotá al menos 10.")
    print("Sí, se lee a mano. Es la parte cara y es la que hace investigación.")
else:
    cuenta = Counter(MI_TIPOLOGIA.values())
    total = sum(cuenta.values())
    print(f"TIPOLOGÍA DE LOS FALSOS POSITIVOS  (n = {total})")
    print("-" * 46)
    for tipo, n in cuenta.most_common():
        print(f"   {tipo:<14}{n:>4}   {n / total:>6.0%}")
    de_etiqueta = cuenta.get("etiqueta", 0)
    if de_etiqueta / total > 0.3:
        print(f"\n>> {de_etiqueta / total:.0%} de tus «errores» son desacuerdos con la etiqueta.")
        print("   Entonces tu F1 no está midiendo la calidad del modelo: está midiendo")
        print("   cuánto coincidís con quien codificó el corpus. Eso hay que decirlo.")

**Tu respuesta:** _(qué te enseñaron los errores sobre tu propia definición de «política»)_

---

## 7. Test 4: el control positivo, el placebo, y dos parientes

Intervenciones hermanas, prestadas del diseño experimental de toda la vida, y
conviene no confundirlas:

- **Control positivo:** aplicá el modelo donde el fenómeno **tiene que**
  aparecer. Si no lo encuentra, tampoco lo estaba encontrando antes.
- **Placebo:** aplicalo donde el fenómeno **no debería** aparecer. Si aparece
  igual, el modelo está detectando otra cosa.
  *Si tu detector de discurso de odio marca el boletín oficial, algo pasa.*
- **Test de invariancia:** cambiá algo que **no debería importar** (el orden de
  las palabras) y mirá si la predicción se mueve. No es un placebo: el fenómeno
  sigue estando ahí.
- **Fuera de dominio:** el mismo fenómeno, en un corpus distinto del de
  entrenamiento. Si ahí el fenómeno está, funciona como un segundo control
  positivo.

El control positivo es el más raro de ver en un paper y casi siempre es el que
más duele. Vamos con todos.

In [ ]:
# --- CONTROL POSITIVO: discursos presidenciales argentinos ------------------
# 123 discursos reales de 2015-2024. Son, por definición, 100 % política.
# Tomamos TODAS sus oraciones de 60 a 220 caracteres (no las primeras N, que
# saldrían casi todas de los primeros discursos del archivo).
def oraciones(texto):
    return [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto) if 60 < len(f.strip()) < 220]


discursos = datos.discursos_casarosada()
fragmentos = [frase for d in discursos for frase in oraciones(d["discurso"])]

# La referencia justa: oraciones del MISMO largo sacadas de las notas de
# política del propio test. El modelo se entrenó con documentos de ~100
# palabras y un fragmento tiene ~20: parte de la caída es el largo, no el país.
oraciones_propias = [s for i in i_test if y[i] == 1 for s in oraciones(notas[i]["texto"])]

marcados = modelo.predict(vec.transform(fragmentos))
recall_oraciones = modelo.predict(vec.transform(oraciones_propias)).mean()
print(f"{len(fragmentos)} fragmentos sacados de {len(discursos)} discursos presidenciales argentinos.")
print("Son 100 % política: el modelo TENDRÍA que marcar el 100 %.")
print(f"Marcó el {marcados.mean() * 100:.0f} %.\n")
print(f"Recall en su test (notas enteras)            : {metricas['recall']:.3f}")
print(f"Recall en oraciones sueltas de sus notas     : {recall_oraciones:.3f}   ({len(oraciones_propias)} oraciones)")
print(f"Recall en oraciones de discursos argentinos  : {marcados.mean():.3f}")
print("\nEl modelo no cambió. Cambió dónde lo medimos.")

### ¿Son los nombres propios? La hipótesis del `TODO 1`, puesta a prueba

En el `TODO 1` escribiste una hipótesis leyendo coeficientes. Ahora la testeamos
con una intervención: **reentrenamos el modelo sin los nombres propios** y
volvemos a correr el control positivo. Si el problema fuera "detecta nombres de
la coyuntura mexicana", sin ellos debería encontrar muchos más discursos.

> **Apostá antes de correr.** Hoy encuentra el 22 %. Sin nombres propios,
> ¿cuánto? ¿El 30 %? ¿El 60 %? ¿El 90 %?

In [ ]:
corpus_sin_np = corpus_ablado["los nombres propios"]
v_np, m_np = entrenar(corpus_sin_np, i_train)

print(f"{'':<34}{'original':>10}{'sin nombres':>13}")
print("-" * 57)
print(f"{'F1 en su propio test':<34}{metricas['f1']:>10.3f}"
      f"{evaluar(v_np, m_np, corpus_sin_np, i_test)['f1']:>13.3f}")
print(f"{'recall en oraciones de sus notas':<34}{recall_oraciones:>10.3f}"
      f"{m_np.predict(v_np.transform(oraciones_propias)).mean():>13.3f}")
print(f"{'discursos argentinos marcados':<34}{marcados.mean():>10.3f}"
      f"{m_np.predict(v_np.transform(fragmentos)).mean():>13.3f}")

In [ ]:
# --- PLACEBO: donde el fenómeno NO debería aparecer --------------------------
# Notas de Deporte y de Espectáculos del test: el modelo no las vio, son del
# mismo corpus y del mismo registro, y no son sobre política.
i_placebo = [i for i in i_test if notas[i]["tema"] in ("Deporte", "Espectaculos")]
pred_placebo = modelo.predict(vec.transform([documentos[i] for i in i_placebo]))
print(f"PLACEBO: {len(i_placebo)} notas de Deporte y Espectáculos que el modelo no vio.")
print(f"Marcó {pred_placebo.sum()} como política ({pred_placebo.mean():.0%}). Son estas:")
for i, p in zip(i_placebo, pred_placebo):
    if p:
        print(f"   [{notas[i]['tema']:<12}] {notas[i]['titular'][:80]}")

# --- TEST DE INVARIANCIA: el mismo texto con las palabras barajadas ---------
# Si el modelo usa el ORDEN de las palabras, barajarlas debería moverlo.
# No es un placebo: el fenómeno sigue estando en el texto.
rng = np.random.default_rng(SEMILLA)


def barajar(texto):
    partes = texto.split()
    rng.shuffle(partes)
    return " ".join(partes)


docs_test = [documentos[i] for i in i_test]
pred_normal = modelo.predict(vec.transform(docs_test))
pred_baraja = modelo.predict(vec.transform([barajar(t) for t in docs_test]))
iguales = (pred_normal == pred_baraja).mean()
print(f"\nINVARIANCIA: predicciones idénticas con el texto barajado: {iguales:.1%}")

# --- FUERA DE DOMINIO: prensa de un país ficticio --------------------------
# Aguasclaras: SINTÉTICO, país ficticio, titulares de 2015-2024 escritos sin
# tildes. Casi todos son de política institucional (sesiones, bloques, leyes,
# protestas): acá el fenómeno SÍ está, así que funciona como un segundo
# control positivo, no como placebo.
ac = datos.prensa_aguasclaras()
tit_ac = [n["titular"] for n in ac]
marcados_ac = modelo.predict(vec.transform(tit_ac))
legislativo = re.compile(r"\b(ley|leyes|sesion|bloque|recinto|comision|diputad\w*|"
                         r"senad\w*|oficialismo|legislatur\w*|camara)\b", re.I)
print(f"\nFUERA DE DOMINIO: {len(tit_ac)} titulares de Aguasclaras (SINTÉTICO, país ficticio).")
print(f"{np.mean([bool(legislativo.search(t)) for t in tit_ac]):.0%} nombran al Congreso, "
      f"una ley, una sesión o un bloque. El modelo marcó el {marcados_ac.mean():.0%} como política.")

### `TODO 4` — ¿qué te dijo cada test?

**(a) El control positivo.** El modelo encuentra alrededor de 6 de cada 10 notas
de política en su propio test; en oraciones sueltas de esas mismas notas, 5 de
cada 10; y en oraciones de discursos donde *todas* son política, 2 de cada 10.
¿Qué parte de la caída es el largo del texto y qué parte es el dominio? Escribí,
en una oración, qué está midiendo realmente. Y decidí: **¿ese modelo sirve para
tu pregunta de investigación?** (No hay una sola respuesta correcta: sirve para
describir *ese* corpus y no sirve para comparar entre corpus. La respuesta es
*para qué*.)

**(b) La hipótesis de los nombres.** Sin nombres propios, el F1 casi no cambia y
el control positivo sube de 22 a 32 %. ¿Tu hipótesis del `TODO 1` sobrevivió,
sobrevivió a medias o se cayó? Reescribila con lo que sabés ahora. *(Fijate que
aun sin nombres, dos de cada tres discursos siguen sin marcarse: ¿qué otra cosa
distingue un discurso presidencial argentino de una nota de prensa mexicana?)*

**(c) El placebo.** Mirá las notas de Deporte y Espectáculos que marcó.
¿Son errores del modelo, errores de la etiqueta, o notas que genuinamente
tocan política? ¿Qué tienen en común?

**(d) El barajado.** Las predicciones son **idénticas**, hasta la última.
¿Por qué era completamente predecible? *Pista: ¿qué es una "bolsa" de palabras?*
Y la parte útil: si este mismo test lo corrés sobre un transformer (clase 7) y
las predicciones **también** te dan idénticas, ¿qué acabás de descubrir sobre ese
modelo, que costó mil veces más?

**Tu respuesta:** _(escribí acá (a), (b), (c) y (d))_

---

## 8. Test 5: los pares mínimos

Los cuatro tests anteriores son **observacionales**: mirás lo que el modelo hace
con los datos que ya tenés. Este es **experimental**: vos fabricás los casos,
cambiando **una sola cosa por vez**, y mirás si el modelo cambia por lo que
debería cambiar.

Es el mismo diseño que un experimento de encuesta con viñetas, y es más
informativo que diez mil casos observacionales, porque vos controlás qué varía.

In [ ]:
PARES = [
    ("negación",
     "El gobierno aprobó la ley de presupuesto",
     "El gobierno no aprobó la ley de presupuesto"),
    ("negación",
     "El senado rechazó el proyecto",
     "El senado no rechazó el proyecto"),
    ("cambio de país",
     "AMLO anunció una reforma al sistema de salud",
     "El presidente de Uruguay anunció una reforma al sistema de salud"),
    ("cambio de país",
     "Morena ganó las elecciones en el estado",
     "El Frente Amplio ganó las elecciones en el departamento"),
    ("cambio de agente",
     "El diputado presentó un proyecto sobre vivienda",
     "El futbolista presentó un proyecto sobre vivienda"),
    ("cambio de agente",
     "El ministro visitó el hospital de la ciudad",
     "El cantante visitó el hospital de la ciudad"),
    # la paráfrasis cambia a propósito TODAS las palabras y deja fijo el
    # contenido: lo único que varía es cómo está dicho
    ("paráfrasis",
     "El congreso debate la reforma electoral",
     "La cámara discute los cambios al sistema de votación"),
    ("sinónimo",
     "El presidente presentó su plan de gobierno",
     "El primer mandatario presentó su plan de gobierno"),
]


def puntaje(texto):
    """El puntaje del modelo para un texto suelto. De 0 a 1."""
    return float(modelo.predict_proba(vec.transform([texto]))[0, 1])


print(f"{'qué cambia':<28}{'A':>7}{'B':>7}{'Δ':>8}")
print("=" * 92)
resultados_pares = []
for tipo, a, b in PARES:
    sa, sb = puntaje(a), puntaje(b)
    resultados_pares.append((tipo, a, b, sa, sb))
    print(f"{tipo:<28}{sa:>7.3f}{sb:>7.3f}{sb - sa:>+8.3f}")
    print(f"   A: {a}")
    print(f"   B: {b}\n")

In [ ]:
# El dibujo de mancuernas: cada par es una línea entre sus dos puntajes.
# Los que se movieron más de 0,15 van en rojo Y con línea llena; los que no,
# en verde y punteados (para que se lea también sin color).
fig, ax = g.figura(9, 4.6, titulo="Pares mínimos: qué mueve el puntaje y qué no",
                   xlabel="puntaje de «esto es sobre política»")
for k, (tipo, a, b, sa, sb) in enumerate(resultados_pares):
    yk = len(resultados_pares) - k
    grande = abs(sb - sa) > 0.15
    ax.plot([sa, sb], [yk, yk], "-" if grande else ":", color=g.ERROR if grande else g.OK,
            lw=2.5, zorder=1)
    ax.plot([sa], [yk], "o", color=g.MODELO, ms=9, zorder=2)
    ax.plot([sb], [yk], "s", color=g.VAL, ms=9, zorder=2)
    ax.text(max(sa, sb) + 0.012, yk, f"{sb - sa:+.2f}", va="center", fontsize=9, color=g.DATOS)
ax.set_yticks(range(1, len(resultados_pares) + 1))
ax.set_yticklabels([t for t, *_ in resultados_pares][::-1], fontsize=10)
ax.axvline(0.5, color=g.DATOS, ls=":", lw=1.5)
ax.text(0.505, 0.5, " umbral 0,5", color=g.DATOS, fontsize=9)
ax.plot([], [], "o", color=g.MODELO, label="oración A")
ax.plot([], [], "s", color=g.VAL, label="oración B")
ax.plot([], [], "-", color=g.ERROR, lw=2.5, label="se movió más de 0,15")
ax.plot([], [], ":", color=g.OK, lw=2.5, label="se movió menos")
ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
ax.set_xlim(0.2, 0.95);

Mirá el resultado con cuidado, porque tiene las dos caras:

- **La negación casi no mueve el puntaje.** "El gobierno aprobó la ley" y "el
  gobierno **no** aprobó la ley" reciben prácticamente el mismo número. Para
  este modelo la negación no existe: es una bolsa de palabras.
- **Decir lo mismo con otras palabras mueve muchísimo.** La paráfrasis de la
  reforma electoral se cae del otro lado del umbral, y cambiar sólo "presidente"
  por "primer mandatario" le saca 0,12.
- **El país, depende del par.** Morena → Frente Amplio baja 0,26; AMLO → "el
  presidente de Uruguay" sube 0,03. Con dos pares no se puede decir nada general
  sobre países: se puede decir que *ese* nombre pesa.

Las tres cosas son coherentes con lo que ya sabemos: el modelo mide
**vocabulario**, y buena parte de ese vocabulario es de una coyuntura.

### `TODO 5` — escribí **tres pares mínimos tuyos**

Elegí una categoría de **tu** investigación (conflicto laboral, género,
migración, populismo, lo que sea) y escribí tres pares que la pongan a prueba.
Uno de cada tipo, por lo menos:

1. Uno donde el modelo **debería** cambiar y querés confirmar que cambia (y
   para qué lado).
2. Uno donde el modelo **no debería** cambiar (cambiás **una** palabra
   irrelevante).
3. Uno **límite**, de esos que a vos mismo te costaría codificar.

*Pista.* El par 3 es el más valioso: si a vos te cuesta decidirlo, tu definición
operativa todavía no está terminada, y eso no lo arregla ningún modelo.

*Mirá los tres que vienen precargados antes de reemplazarlos:* el par de
control ("mañana" → "tarde") **ya falla**, y el "debería cambiar" se mueve…
para el lado contrario. Leé el signo, no sólo el tamaño.

In [ ]:
# TODO 5: reemplazá estos tres pares por los tuyos.
MIS_PARES = [
    ("debería cambiar", "Los trabajadores anunciaron un paro general",
                        "Los trabajadores anunciaron su balance anual"),
    ("no debería cambiar", "El gremio se reunió el martes por la mañana",
                           "El gremio se reunió el martes por la tarde"),
    ("caso límite", "La empresa y el sindicato firmaron el acta",
                    "La empresa y el sindicato no llegaron a un acuerdo"),
]

assert len(MIS_PARES) >= 3, (
    "Hacen falta al menos tres pares. Escribirlos ES el ejercicio: "
    "diseñar el test es más difícil (y más útil) que correrlo.")

print(f"{'qué cambia':<22}{'A':>7}{'B':>7}{'Δ':>8}   ¿se comportó como esperabas?")
print("-" * 88)
for tipo, a, b in MIS_PARES:
    sa, sb = puntaje(a), puntaje(b)
    delta = sb - sa
    if tipo.startswith("debería") and abs(delta) < 0.05:
        veredicto = "NO — esperabas movimiento y no se movió"
    elif tipo.startswith("no debería") and abs(delta) > 0.05:
        veredicto = "NO — no esperabas movimiento y se movió"
    elif tipo.startswith("debería"):
        veredicto = "se movió: ¿para el lado que esperabas?"
    else:
        veredicto = "parece que sí (miralo igual)"
    print(f"{tipo:<22}{sa:>7.3f}{sb:>7.3f}{delta:>+8.3f}   {veredicto}")

**Tu respuesta:** _(qué aprendiste de tus tres pares)_

---

## 9. La auditoría de sesgo, hecha

Acá **no discutimos** el sesgo: lo **medimos**, con un procedimiento que
produce una tabla que se puede pegar en un apéndice.

El procedimiento es el de la sección anterior aplicado sistemáticamente:
una plantilla de oración con un hueco para el sujeto, una lista de sujetos que
varían en una dimensión, y los puntajes ordenados.

**Tres sesgos que no son lo mismo**, y confundirlos lleva a soluciones absurdas:

| | qué es | cómo se arregla |
|---|---|---|
| **del mundo** | la realidad que medís es desigual | no se "arregla": se estudia. Si tu modelo la refleja, el problema es qué vas a hacer con él |
| **del corpus** | tu muestra no representa el mundo | con muestreo, con ponderación, con más datos de donde faltan |
| **introducido** | lo pusiste vos: el umbral, la loss, la métrica que elegiste reportar | revisando esas decisiones, que son cinco y todas discutibles |

Balancear los datos ayuda con el segundo. No toca el primero. Y a veces
distorsiona la estimación de prevalencia, que era justo lo que querías medir. El
modelo de hoy, sin ir más lejos, entrena con `class_weight="balanced"`: mirá qué
pasa si lo usás para **contar** notas de política.

In [ ]:
# Si usás el modelo para ESTIMAR PREVALENCIA contando predicciones:
m_ = g.metricas_binarias(y[i_test], pred_normal)
cuenta = pred_normal.mean()
fpr = m_["fp"] / (m_["fp"] + m_["vn"])
print(f"Proporción real de notas de política en el test : {y[i_test].mean():.1%}")
print(f"Proporción que marca el modelo (contando)        : {cuenta:.1%}   "
      f"({m_['fp']} falsos positivos, {m_['fn']} falsos negativos)")
print(f"Corrigiendo con el recall y la tasa de FP         : "
      f"{(cuenta - fpr) / (m_['recall'] - fpr):.1%}")
print("\nContar predicciones no es medir prevalencia cuando los dos errores no se")
print("compensan. La corrección (classify-and-count ajustado) necesita recall y tasa")
print("de falsos positivos medidos en una muestra validada APARTE: acá la hicimos")
print("sobre el mismo test, que es circular (por eso da EXACTO el valor real), sólo")
print("para mostrar la cuenta.")

In [ ]:
PLANTILLA = "{} reclamó frente a la sede del ministerio"

SUJETOS = [
    ("género", "los trabajadores"), ("género", "las trabajadoras"),
    ("género", "los médicos"), ("género", "las médicas"),
    ("género", "los empresarios"), ("género", "las empresarias"),
    ("ocupación", "los docentes"), ("ocupación", "los policías"),
    ("ocupación", "los taxistas"), ("ocupación", "los estudiantes"),
    ("ocupación", "los enfermeros"),
    ("origen", "los vecinos"), ("origen", "los migrantes"),
    ("origen", "los venezolanos"), ("origen", "los mexicanos"),
    ("origen", "los campesinos"), ("origen", "los indígenas"),
]

vocab_set = set(vocabulario)
filas = []
for dim, sujeto in SUJETOS:
    frase = PLANTILLA.format(sujeto)
    fuera = [w for w in sujeto.split() if w not in vocab_set]
    filas.append({"dimension": dim, "sujeto": sujeto, "score": puntaje(frase),
                  "fuera": fuera})
filas.sort(key=lambda f: -f["score"])

print(f'PLANTILLA:  «[SUJETO] {PLANTILLA.format("").strip()}»\n')
print(f"{'sujeto':<22}{'dimensión':<12}{'puntaje':>9}   ¿lo conoce el modelo?")
print("-" * 74)
for f in filas:
    conoce = "sí" if not f["fuera"] else "NO: " + ", ".join(f["fuera"])
    print(f"{f['sujeto']:<22}{f['dimension']:<12}{f['score']:>9.3f}   {conoce}")

dif_max = filas[0]["score"] - filas[-1]["score"]
print(f"\nDIFERENCIA MÁXIMA: {dif_max:.3f}   "
      f"(«{filas[0]['sujeto']}» {filas[0]['score']:.3f} contra "
      f"«{filas[-1]['sujeto']}» {filas[-1]['score']:.3f})")
print(f"Es decir: {dif_max * 100:.0f} puntos sobre 100. (No lo reportes como «tantas veces")
print("más alto»: un cociente entre puntajes bajos y no calibrados exagera.)")

In [ ]:
fig, ax = g.figura(9, 5.6,
                   titulo="La misma oración, distinto sujeto",
                   xlabel="puntaje de «esto es sobre política»")
etiquetas = [f["sujeto"] + (" (?)" if f["fuera"] else "") for f in filas][::-1]
valores = [f["score"] for f in filas][::-1]
fuera_de_vocab = [bool(f["fuera"]) for f in filas][::-1]
barras = ax.barh(etiquetas, valores,
                 color=[g.VERDAD if f else g.MODELO for f in fuera_de_vocab])
for b, f in zip(barras, fuera_de_vocab):
    if f:
        b.set_hatch("//")
        b.set_edgecolor("white")
ax.axvline(np.mean(valores), color=g.DATOS, ls=":", lw=1.5)
ax.text(np.mean(valores), len(valores) - 0.4, " promedio", color=g.DATOS, fontsize=9)
ax.text(0.98, 0.03, "en gris, rayado y con (?): el modelo nunca vio esa palabra",
        transform=ax.transAxes, ha="right", color=g.VERDAD, fontsize=10,
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"));

### Antes de interpretar: **la ceguera no es equidad**

Fijate en las barras grises (las rayadas, con «(?)»). Varios sujetos reciben **exactamente** el mismo
puntaje, hasta el último decimal. Eso no quiere decir que el modelo los trate
igual de bien: quiere decir que **nunca vio esa palabra**, y que lo único que
está leyendo es el artículo (*los* / *las*) y el resto de la oración.

Un puntaje idéntico se reporta distinto según de dónde venga. Y hay una
consecuencia práctica: **un modelo no puede ser justo con un grupo que no está
en sus datos.** Ausencia no es neutralidad.

In [ ]:
# ¿La diferencia por género se distingue del azar? Y, sobre todo, ¿de dónde sale?
PARES_GENERO = [
    ("los trabajadores", "las trabajadoras"), ("los médicos", "las médicas"),
    ("los empresarios", "las empresarias"), ("los docentes", "las docentes"),
    ("los enfermeros", "las enfermeras"), ("los vecinos", "las vecinas"),
    ("los estudiantes", "las estudiantes"), ("los jubilados", "las jubiladas"),
    ("los abogados", "las abogadas"), ("los periodistas", "las periodistas"),
]
sm = np.array([puntaje(PLANTILLA.format(m)) for m, _ in PARES_GENERO])
sf = np.array([puntaje(PLANTILLA.format(f)) for _, f in PARES_GENERO])
dif = sf - sm


def peso_de(palabra):
    return float(pesos[vec.vocabulary_[palabra]]) if palabra in vec.vocabulary_ else 0.0


# Antes de la sigmoide (en "logit"), la diferencia de cada par es EXACTAMENTE
#   peso(las) − peso(los)  +  peso(sustantivo femenino) − peso(masculino)
w_articulo = peso_de("las") - peso_de("los")
print(f"{'par':<36}{'Δ puntaje':>10}{'Δ logit: artículo':>19}{'+ sustantivo':>14}")
print("-" * 79)
for (m, f), d in zip(PARES_GENERO, dif):
    w_sust = peso_de(f.split()[1]) - peso_de(m.split()[1])
    print(f"{m + ' / ' + f:<36}{d:>+10.3f}{w_articulo:>+19.3f}{w_sust:>+14.3f}")

# Test de signos EXACTO: con 10 pares hay 2**10 = 1024 patrones posibles de
# signos, así que los enumeramos todos en vez de sortear.
patrones = np.array(list(itertools.product([-1, 1], repeat=len(dif))))
nulo = (patrones * dif).mean(axis=1)
p_valor = float((np.abs(nulo) >= abs(dif.mean()) - 1e-12).mean())

print(f"\nDiferencia media (femenino − masculino): {dif.mean():+.4f}")
print(f"p exacto (los {len(patrones)} patrones de signo)   : {p_valor:.4f}  "
      f"(el mínimo posible es {2 / len(patrones):.4f})")
print("\nOJO antes de celebrar el p: las 10 diferencias son positivas porque las 10")
print(f"cambian el MISMO artículo, que suma {w_articulo:+.3f} en todas. No son 10 evidencias")
print("independientes: es un solo coeficiente repetido 10 veces (pseudo-replicación).")
print("El test no agrega nada que no diga ya la columna del artículo.")

In [ ]:
# ¿De dónde sale esa diferencia? Miremos los pesos de las dos palabras que
# cambian. Spoiler: no cambia sólo el sustantivo.
for palabra in ["los", "las", "médicos", "médicas", "docentes", "trabajadoras"]:
    if palabra in vocab_set:
        w = pesos[list(vocabulario).index(palabra)]
        print(f"   {palabra:<14}peso {w:+.4f}")
    else:
        print(f"   {palabra:<14}NO ESTÁ en el vocabulario (peso efectivo 0)")
print("\nBuena parte del «efecto de género» que acabás de medir sale del ARTÍCULO,")
print("no del sustantivo: en 6 de los 10 pares, TODO sale del artículo. Reportar «el")
print("modelo puntúa más alto a las mujeres» sin decir esto sería técnicamente cierto")
print("y sustantivamente engañoso. Lo reportable es la tabla de arriba, descompuesta.")

### `TODO 6` (el de criterio, el más importante del notebook)

Mirá la tabla y el gráfico y contestá las tres, por escrito y completas:

**(a) ¿La diferencia es grande *en términos prácticos*, no sólo estadísticos?**
Traducila a algo concreto: si aplicás este modelo a 10.000 notas con un umbral
de 0,5, ¿cuántas notas de más o de menos marcarías según qué sujeto aparece?
*(Lo que vas a encontrar: con esta plantilla ningún sujeto llega a 0,5, así que
en esta oración la diferencia práctica es cero decisiones. ¿Eso cierra la
discusión? Pensá en las notas reales que están cerca del umbral.)*

**(b) ¿De dónde puede venir?** Nombrá las tres explicaciones posibles (sesgo del
mundo, del corpus, introducido) y decí **cómo distinguirías entre ellas
empíricamente**. No con una opinión: con un dato que podrías ir a buscar.

**(c) ¿Cambia tu conclusión según el uso?** Escribí dos frases: una para el caso
en que el modelo **mide** (contás cuántas notas de política hay en un corpus) y
otra para el caso en que el modelo **decide** (priorizás qué reclamo atiende
primero una oficina pública). ¿La misma tabla justifica lo mismo en los dos
casos?

> **La advertencia que va en el paper, y que es parte del rigor:** una diferencia
> de puntaje **no prueba daño**. Prueba una asociación adentro del modelo. Para
> afirmar daño hay que mostrar el uso concreto, quién decide con eso, y a quién
> le cae el error.

**Tu respuesta:** _(escribí acá (a), (b) y (c))_

---

## 10. Reproducibilidad: la función que te llevás

Nada de esto es difícil. Es disciplina, y hay que hacerla **desde el día uno**,
porque después no se puede reconstruir: nadie se acuerda en diciembre de qué
versión de la biblioteca tenía en marzo.

La celda de abajo genera sola el bloque que va al apéndice. **Copiala a todos
tus proyectos.**

In [ ]:
def bloque_de_reproducibilidad(semilla, modelo=None, datos_descriptos="",
                               n_casos=None, textos=None):
    """
    Arma el bloque de reproducibilidad listo para pegar en el apéndice.

    Copiá esta función a tus proyectos y llamala al final del análisis.
    """
    lineas = ["=" * 72, "BLOQUE DE REPRODUCIBILIDAD", "=" * 72]
    lineas.append(f"Fecha de la corrida     : {date.today().isoformat()}")
    lineas.append(f"Semilla                 : {semilla}")
    lineas.append(f"Python                  : {platform.python_version()} "
                  f"({platform.system()})")

    for nombre in ["numpy", "sklearn", "torch", "matplotlib", "pandas"]:
        try:
            mod = __import__(nombre)
            lineas.append(f"{nombre:<24}: {getattr(mod, '__version__', 'sin versión')}")
        except ImportError:
            lineas.append(f"{nombre:<24}: no instalado")

    if datos_descriptos:
        lineas.append(f"Datos                   : {datos_descriptos}")
    if n_casos is not None:
        lineas.append(f"Casos                   : {n_casos}")
    if textos is not None:
        # El hash de los datos: si el corpus cambia, este número cambia,
        # y te enterás en vez de publicar dos cosas distintas con el mismo nombre.
        h = hashlib.sha256("\n".join(textos).encode("utf-8")).hexdigest()
        lineas.append(f"Hash del corpus (sha256): {h[:16]}…")
    if modelo is not None:
        coef = getattr(modelo, "coef_", None)
        if coef is not None:
            h = hashlib.sha256(np.ascontiguousarray(coef).tobytes()).hexdigest()
            lineas.append(f"Hash de los pesos       : {h[:16]}…")
            lineas.append(f"Parámetros del modelo   : {coef.size + 1}")
    lineas.append("=" * 72)
    return "\n".join(lineas)


print(bloque_de_reproducibilidad(
    SEMILLA, modelo=modelo,
    datos_descriptos=("Spanish Fake News Corpus 2.0 (CC BY 4.0): 971 notas recolectadas "
                      "en ene-jul 2018 + 500 en nov-2020 a mar-2021"),
    n_casos=len(notas), textos=documentos))

### `TODO 7` — el ejercicio que casi siempre encuentra algo

**(a)** Andá a **Entorno de ejecución → Reiniciar y ejecutar todo** (o
`Kernel → Restart & Run All`), corré este notebook entero desde cero y
compará: **¿te dieron *exactamente* los mismos números?**

Deberían, salvo dos cosas: la fecha del bloque de reproducibilidad, que cambia
sola, y la celda del sorteo de abajo, que dejamos **suelta a propósito** (es la
parte (b)). Todas las demás fuentes de azar de este notebook están fijadas. Encontrá **dónde** se fija
cada una: hay por lo menos cinco (pista: buscá `SEMILLA`, `random_state` y
`default_rng`). Y ojo con lo que *no* garantiza la semilla: en otra máquina o con
otra versión de scikit-learn los números pueden cambiar un poco. Por eso el
bloque de arriba anota las versiones.

**(b)** La celda de abajo tiene una fuente de azar **suelta, puesta a
propósito**: elige cinco falsos positivos "al azar" para leer en voz alta.
Correla dos veces seguidas. Después arreglala para que dé siempre los mismos
cinco, **sin** tocar el resto del notebook.

**Ese descubrimiento, hecho con las manos, enseña más que cualquier slide.**
Y es exactamente lo que le va a pasar a quien intente reproducir tu paper.

In [ ]:
# TODO 7 (b): esta celda da un resultado distinto cada vez que la corrés.
# ¿Por qué? Arreglala.
sorteo = np.random.default_rng()
elegidos = sorteo.choice(len(falsos_positivos), size=5, replace=False)
print("Cinco falsos positivos para leer en voz alta:")
for j in sorted(elegidos):
    k, i = falsos_positivos[j]
    print(f"   [{proba[k]:.2f}] {notas[i]['titular'][:80]}")

**Tu respuesta:** _(¿dieron iguales? ¿dónde se fija cada fuente de azar? ¿cómo arreglaste la celda del sorteo?)_

---

## 11. La ficha del modelo

Una **ficha de modelo** (*model card*) es media página que acompaña al
instrumento y contesta lo que cualquiera necesita saber antes de usarlo: qué
mide, con qué se entrenó, cómo se comporta por subgrupo, qué limitaciones tiene
y **para qué NO hay que usarlo**.

Es lo mismo que hacés cuando publicás una escala de actitudes. La novedad es
sólo que ahora también hay que hacerlo para un clasificador.

In [ ]:
pc_sin_np = m_np.predict(v_np.transform(fragmentos)).mean()
np_pareada = pareadas["los nombres propios"]

FICHA = f"""
# Ficha del modelo — clasificador «nota sobre política»

**Versión.** 1.0 · {date.today().isoformat()} · semilla {SEMILLA}

## Qué mide
Estima si una nota de prensa está codificada con el tema «Política» en el
Spanish Fake News Corpus 2.0. NO mide "contenido político" en general:
mide acuerdo con esa codificación, sobre ese corpus.

## Con qué se entrenó
{len(i_train)} notas de prensa digital en castellano (México, España, Argentina,
Colombia, Chile y otros). El corpus junta dos recolecciones: una de enero a
julio de 2018 (971 notas) y otra de noviembre de 2020 a marzo de 2021 (500).
Bolsa de palabras binaria ({len(vocabulario)} términos) + regresión logística
con class_weight='balanced'.

## Desempeño
Sobre {len(i_test)} casos de test: precisión {metricas['precision']:.2f},
recall {metricas['recall']:.2f}, F1 {metricas['f1']:.2f},
accuracy {metricas['accuracy']:.2f} contra un piso de {piso:.2f}.
F1 según la semilla de partición (25 particiones): {efes.mean():.2f} ± {efes.std():.3f} (desvío).

## Desempeño por subgrupo
Auditoría de {len(SUJETOS)} sujetos en tres dimensiones sobre una plantilla fija.
Diferencia máxima de puntaje: {dif_max:.3f}. La diferencia por género sale en
buena parte del artículo («los»/«las»), no del sustantivo.
{sum(1 for f in filas if f['fuera'])} de {len(filas)} sujetos contienen palabras
fuera del vocabulario: para esos, el modelo es ciego, no neutral.

## Limitaciones conocidas
1. Un tercio de sus 20 términos de más peso son nombres propios de la
   política mexicana alrededor de 2018. Sacarlos baja el F1 {abs(np_pareada.mean()):.3f}
   en promedio (25 particiones; poco, y el intervalo roza el cero) y sube el control positivo de
   {marcados.mean() * 100:.0f} a {pc_sin_np * 100:.0f} %: explican una parte de la falla, no toda.
2. Aplicado a oraciones de discursos presidenciales argentinos (100 % política)
   marca sólo el {marcados.mean() * 100:.0f} %, contra {recall_oraciones * 100:.0f} % en oraciones
   del mismo largo de su propio corpus. No transfiere fuera de su dominio.
3. En notas de Deporte y Espectáculos marca {pred_placebo.mean() * 100:.0f} % como política,
   en su mayoría por nombres de políticos.
4. Es una bolsa de palabras: ignora el orden y la negación. Con el texto
   barajado da {iguales:.0%} de predicciones idénticas.
5. Contar sus predicciones subestima la prevalencia ({cuenta:.1%} contra
   {y[i_test].mean():.1%} en el test): para estimar proporciones hay que corregir.
6. La etiqueta la produjo otro equipo con su propia definición operativa.

## Usos NO recomendados
- Comparar niveles de "contenido político" entre corpus, países o épocas.
- Cualquier decisión sobre casos individuales o sobre personas.
- Uso sin una muestra nueva validada a mano en el dominio de destino.

## Cómo citar y reproducir
Código, datos y bloque de reproducibilidad en el repositorio del curso.
"""
print(FICHA)

### `TODO 8` (final) — la ficha de **tu** modelo

Copiá la plantilla de arriba y completala para **un** análisis tuyo: el de tu
tesis, el de tu proyecto, o el que hiciste en la clase 9. Media página.

Si hay campos que no podés completar, **no los borres**: escribí "no medido" y
dejalo a la vista. Un campo que dice "no medido" es honesto y se puede discutir.
Un campo ausente parece que no hacía falta.

**Este es el entregable del curso.** Si te llevás una sola cosa de estas diez
clases, que sea la capacidad de escribir esta media página sobre cualquier
modelo que uses o que te muestren.

**Tu ficha:** _(escribila acá)_

---

## 12. Qué te llevás

**Las tres cosas de hoy:**

1. **Desempeño no es validez.** La métrica te dice que el modelo reproduce tus
   etiquetas. No te dice nada sobre si tus etiquetas miden tu concepto. Un
   modelo con F1 0,68 en su test encontró el 22 % de un corpus que era 100 %
   positivo, y el modelo no falló ni una vez: hizo exactamente lo que le
   pedimos.
2. **La validez se demuestra con intervenciones, no con métricas.** Ablación,
   placebo, control positivo, pares mínimos, validación externa. Y antes de
   interpretar cualquier diferencia, **medí su error, y el correcto**: si
   comparás dos versiones sobre los mismos datos, el error es el de la
   diferencia pareada, no el rango del número suelto.
3. **El sesgo se mide con un procedimiento y se reporta con cuidado.** Una
   diferencia de puntaje prueba una asociación en el modelo, no un daño. Y un
   puntaje idéntico muchas veces es ceguera, no equidad.

**El vocabulario nuevo** (también en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *construct validity* | validez de constructo — ¿la cosa que medís es la cosa de la que hablás? |
| *ablation* | ablación — sacarle algo al modelo o al corpus y ver cuánto cae |
| *placebo test* | test de placebo — aplicarlo donde el fenómeno no debería estar |
| *positive control* | control positivo — aplicarlo donde el fenómeno tiene que estar |
| *minimal pair* | par mínimo — dos textos idénticos salvo por una cosa |
| *model card* | ficha del modelo — la media página que acompaña al instrumento |
| *shortcut learning* | atajo — el modelo aprendió algo que separa las clases pero no es el concepto |
| *out-of-distribution* | fuera de dominio — datos que no se parecen a los del entrenamiento |

**En tu propia investigación, desde ya.** Agarrá el último análisis que
hiciste con un modelo (tuyo o de otro) y correle **dos** cosas: el control
positivo (aplicalo donde el fenómeno tiene que estar) y tres pares mínimos
escritos a mano. Son veinte minutos. Si sobrevive, tenés un párrafo nuevo para
la sección de métodos. Si no sobrevive, te enteraste vos antes que el revisor 2.

**Lo que queda abierto.** Nada, y esa es la gracia: de acá en adelante el
programa lo escribís vos. La sugerencia es una sola y es concreta: **no hagas
otro curso, hacé un proyecto chico con tus datos.** Uno que puedas terminar.
Con su semilla, su ficha y su párrafo de métodos.

---

### Para seguir

Las referencias están en `referencias.md`. Las de ciencias sociales van primero,
como siempre: lo que importa no es el paper de ML, es el trabajo que muestra qué
se puede y qué no se puede afirmar con esto.

> Durante diez clases aprendimos a dibujar. Hoy aprendimos a pararnos frente a
> un dibujo y preguntar: **¿de qué es esto un dibujo?**